# Data Assembly Pipeline: CalEnviroScreen × California Cancer Registry
## Merging Environmental, Socioeconomic, and Health Data at the MSSA Level

---

### Purpose

This notebook constructs the analytical dataset used in the accompanying
analysis pipeline. It merges five publicly available data sources into a
single MSSA-level (Medical Service Study Area) file containing:

- **Age-adjusted lung cancer incidence rates (AAIR)** from the California
  Cancer Registry (via California Health Maps).
- **Environmental pollution and socioeconomic indicators** from
  CalEnviroScreen 4.0, aggregated from census tracts to MSSAs using
  population-weighted means.
- **MSSA geographic area** (square miles) from the OSHPD Medical Service
  Study Areas shapefile.
- **Smoking prevalence** from the CDC PLACES / Health Atlas dataset,
  linked at the census-tract level and aggregated to MSSAs.

### Data Flow

```
CalEnviroScreen 4.0 (tract-level)
     │
     ├── merge ← Smoking prevalence (tract-level, CDC PLACES)
     │
     ├── merge ← Geography crosswalk (tract → MSSA)
     │
     └── population-weighted aggregation → MSSA-level pollution + smoking
                                                  │
California Health Maps (MSSA-level AAIR) ─── merge ┤
                                                  │
MSSA area (sq mi) ────────────────────────── merge ┘
                                                  │
                                                  ▼
                                          Final merged CSV
```

### Output

A single CSV file (`healthmaps_with_ces4_pollution_plus_area.csv`)
containing one row per MSSA × sex × cancer combination, filtered to
**lung cancer** and the **10-year reporting period**.

### Reproducibility Notes

- All file paths are specified as relative paths in the configuration
  cell below; update them to match your local directory structure.
- Census tract identifiers are normalised to plain digit strings to
  ensure consistent joins across datasets with varying formatting.
- Population-weighted aggregation falls back to unweighted means when
  all tract-level population weights are zero or missing.

## 1. Configuration and Helper Functions

The cell below defines:

1. **File paths** — relative paths to each input dataset and the output
   CSV. Update these to match your directory layout.
2. **`canon_tract()`** — normalises census-tract identifiers to plain
   digit strings (handles float, int, and string representations).
3. **`pick_pollution_columns()`** — heuristically selects numeric
   CalEnviroScreen indicator columns while excluding identifiers,
   percentile fields, and composite scores.
4. **`pop_weighted_mean()`** — computes population-weighted means within
   each MSSA, with a fallback to unweighted means when weights are
   unavailable.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ══════════════════════════════════════════════════════════════════════════
# FILE PATHS — update these to match your local directory structure
# ══════════════════════════════════════════════════════════════════════════
CALENVIRO_PATH  = Path("./data/calenviro2025.csv")
HEALTHMAPS_PATH = Path("./data/californiahealthmaps_mssa_all.csv")
XWALK_PATH      = Path("./data/geography-crosswalk.csv")
MSSA_AREA_PATH  = Path("./data/Medical_Service_Study_Areas.csv")
SMOKING_PATH    = Path("./data/health-atlas-2026-03-24-49fe0d25.csv")
OUT_PATH        = Path("./data/healthmaps_with_ces4_pollution_plus_area.csv")


# ── Helpers ──────────────────────────────────────────────────────────────

def canon_tract(val) -> str:
    """
    Normalise a census-tract identifier to a plain string of digits.

    Census-tract IDs appear variously as integers, floats (e.g. 6001400100.0),
    or zero-padded strings across different data sources. This function
    coerces all representations to a consistent digit-only string to enable
    reliable joins.
    """
    try:
        return str(int(float(val)))
    except Exception:
        return str(val).strip()


def pick_pollution_columns(df: pd.DataFrame) -> list:
    """
    Heuristically select CalEnviroScreen indicator columns for aggregation.

    Retains numeric columns that are NOT identifiers (e.g. Census Tract,
    Latitude) and do NOT contain percentile, range, or composite-score
    keywords. This isolates the raw pollution and environmental-health
    indicator values suitable for population-weighted aggregation.
    """
    id_like = {
        'Census Tract', 'Longitude', 'Latitude', 'ZIP', 'Total Population',
        'California County', 'Approximate Location'
    }
    exclude_substrings = [
        'Pctl', 'Percentile', 'Percentile Range', 'Range',
        'SB 535', 'Score', 'Pop. Char.'
    ]

    keep = []
    for col in df.columns:
        if col in id_like:
            continue
        lower = col.lower()
        if any(s.lower() in lower for s in exclude_substrings):
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            keep.append(col)
    return keep


def pop_weighted_mean(group, value_cols, weight_col='Total Population'):
    """
    Compute population-weighted means for multiple columns within a group.

    For each value column, rows with NaN in either the value or the weight
    are excluded. If total weight is zero (all tracts have zero or missing
    population), the function falls back to an unweighted mean so that no
    MSSA is silently dropped.

    Parameters
    ----------
    group : DataFrame
        Subset of tract-level rows belonging to one MSSA.
    value_cols : list of str
        Columns to aggregate.
    weight_col : str
        Column containing population weights.

    Returns
    -------
    pd.Series with one weighted mean per value column.
    """
    results = {}
    for col in value_cols:
        vals = group[col].values.astype(float)
        weights = group[weight_col].values.astype(float)

        # Mask rows where either value or weight is NaN
        mask = ~(np.isnan(vals) | np.isnan(weights))
        vals = vals[mask]
        weights = weights[mask]

        total_w = weights.sum()
        if total_w > 0 and len(vals) > 0:
            results[col] = np.average(vals, weights=weights)
        elif len(vals) > 0:
            results[col] = np.nanmean(vals)       # fallback: unweighted
        else:
            results[col] = np.nan
    return pd.Series(results)

print("Configuration loaded. Paths are relative — update if needed.")

Configuration loaded. Paths are relative — update if needed.


## 2. Data Loading and Key Normalisation

Each dataset is loaded and its join key is normalised:

| Dataset | Key field | Normalisation |
|---------|-----------|---------------|
| CalEnviroScreen 4.0 | `Census Tract` | `canon_tract()` → `tract_key` |
| Geography crosswalk | `Census_Tract` | `canon_tract()` → `tract_key` |
| California Health Maps | `AreaID` | Strip whitespace |
| MSSA area shapefile | `MSSAID` | Strip whitespace |
| Smoking (CDC PLACES) | `GEOID` | Filter to California (FIPS 06), then `canon_tract()` |

The Health Maps data is filtered to the **10-year reporting period**
(`Years == '10yr'`) to match the study's aggregation window.

In [2]:
# ── Load all five input datasets ─────────────────────────────────────────
cal = pd.read_csv(CALENVIRO_PATH)                          # CalEnviroScreen 4.0
chm = pd.read_csv(HEALTHMAPS_PATH)                         # California Health Maps
chm = chm[chm['Years'] == '10yr']                          # Retain 10-year aggregate only
xw  = pd.read_csv(XWALK_PATH)                              # Census tract → MSSA crosswalk
msa = pd.read_csv(MSSA_AREA_PATH)                          # MSSA area (sq mi)
smk = pd.read_csv(SMOKING_PATH, dtype={'GEOID': str})      # Smoking prevalence (CDC PLACES)

print(f"CalEnviroScreen:       {cal.shape[0]:>7,} tracts  × {cal.shape[1]} cols")
print(f"Health Maps (10yr):    {chm.shape[0]:>7,} rows    × {chm.shape[1]} cols")
print(f"Crosswalk:             {xw.shape[0]:>7,} rows    × {xw.shape[1]} cols")
print(f"MSSA area:             {msa.shape[0]:>7,} rows    × {msa.shape[1]} cols")
print(f"Smoking (raw):         {smk.shape[0]:>7,} tracts  × {smk.shape[1]} cols")

# ── Normalise join keys ──────────────────────────────────────────────────
cal['tract_key'] = cal['Census Tract'].apply(canon_tract)
xw['tract_key']  = xw['Census_Tract'].apply(canon_tract)

xw['MSSA_ID']  = xw['MSSA_ID'].astype(str).str.strip()
chm['AreaID']  = chm['AreaID'].astype(str).str.strip()
msa['MSSAID']  = msa['MSSAID'].astype(str).str.strip()

# ── Prepare smoking data: restrict to California (FIPS state code 06) ───
smk = smk[smk['GEOID'].str.startswith('06')].copy()
smk['tract_key'] = smk['GEOID'].apply(canon_tract)
smk = smk.rename(columns={'csmoking_crudeprev': 'Smoking'})

print(f"\nSmoking (CA only):     {smk.shape[0]:>7,} tracts")
print("Key normalisation complete.")

CalEnviroScreen:         8,035 tracts  × 58 cols
Health Maps (10yr):     17,886 rows    × 42 cols
Crosswalk:               8,036 rows    × 8 cols
MSSA area:               9,129 rows    × 17 cols
Smoking (raw):          83,414 tracts  × 2 cols

Smoking (CA only):       9,060 tracts
Key normalisation complete.


## 3. Tract-to-MSSA Aggregation (Population-Weighted)

CalEnviroScreen indicators are reported at the census-tract level, but the
analysis operates at the MSSA level. This section:

1. **Identifies pollution indicator columns** using the heuristic filter
   (`pick_pollution_columns`), excluding identifiers, percentiles, and
   composite scores.
2. **Merges tract-level smoking prevalence** from CDC PLACES onto the
   CalEnviroScreen tract data.
3. **Links each census tract to its parent MSSA** via the geography
   crosswalk.
4. **Aggregates to the MSSA level** using population-weighted means, so
   that tracts with larger populations contribute proportionally more to
   the MSSA-level estimate.

The resulting DataFrame has one row per MSSA and one column per indicator.

In [3]:
# ── Identify pollution indicator columns to aggregate ────────────────────
pollution_cols = pick_pollution_columns(cal)
print(f"Pollution/indicator columns selected: {len(pollution_cols)}")
print(f"  Examples: {pollution_cols[:8]}")

# ── Select CalEnviro columns: tract key + population weight + indicators ─
cal_sel = cal[['tract_key', 'Total Population'] + pollution_cols].copy()

# ── Merge tract-level smoking prevalence ─────────────────────────────────
cal_sel = cal_sel.merge(smk[['tract_key', 'Smoking']], on='tract_key', how='left')
n_smoking = cal_sel['Smoking'].notna().sum()
print(f"Smoking data matched: {n_smoking}/{len(cal_sel)} tracts "
      f"({n_smoking/len(cal_sel)*100:.1f}%)")

# ── Map tracts to MSSAs via the geography crosswalk ──────────────────────
cal_mapped = cal_sel.merge(xw[['tract_key', 'MSSA_ID']], on='tract_key', how='inner')
n_mapped = cal_mapped['MSSA_ID'].nunique()
print(f"Tracts successfully mapped to MSSAs: {len(cal_mapped)} "
      f"({n_mapped} unique MSSAs)")

# ── Population-weighted aggregation to MSSA level ────────────────────────
agg_cols = pollution_cols + ['Smoking']
pollution_by_mssa = (
    cal_mapped
    .groupby('MSSA_ID')
    .apply(lambda g: pop_weighted_mean(g, agg_cols, weight_col='Total Population'))
    .reset_index()
)
print(f"Aggregated to {len(pollution_by_mssa)} MSSAs")

# ── Rename columns: add CES_ prefix for CalEnviroScreen indicators,
#    rename Smoking to PerSmoking for consistency with other percentage vars
rename_map = {c: f"CES_{c.replace(' ', '_')}" for c in pollution_cols}
rename_map['Smoking'] = 'PerSmoking'
pollution_by_mssa = pollution_by_mssa.rename(columns=rename_map)

Pollution/indicator columns selected: 22
  Examples: ['Ozone', 'PM2.5', 'Diesel PM', 'Drinking Water', 'Lead', 'Pesticides', 'Tox. Release', 'Traffic']
Smoking data matched: 6823/8035 tracts (84.9%)
Tracts successfully mapped to MSSAs: 8034 (542 unique MSSAs)
Aggregated to 542 MSSAs


/tmp/ipykernel_271824/3630642608.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pop_weighted_mean(g, agg_cols, weight_col='Total Population'))


## 4. Final Merge and Output

The MSSA-level pollution/smoking indicators are merged onto the California
Health Maps AAIR data via the shared MSSA identifier (`AreaID` /
`MSSA_ID`). MSSA geographic area (square miles) is also joined to support
population-density calculations downstream.

The merged dataset is then:
- **Filtered to lung cancer** (`Cancer == 'Lung'`).
- **Stripped of columns** not needed for analysis (`Counties`, `Cities`,
  `URL`).
- **Saved** as a single CSV for use by the analysis notebook.

In [4]:
# ── Prepare MSSA area lookup (one row per MSSA) ──────────────────────────
mssa_area = (
    msa[['MSSAID', 'ASQMI']]
    .drop_duplicates(subset=['MSSAID'])
    .rename(columns={'MSSAID': 'AreaID', 'ASQMI': 'MSSA_Area_SqMi'})
)

# ── Merge pollution indicators onto Health Maps AAIR data ────────────────
merged = chm.merge(
    pollution_by_mssa, left_on='AreaID', right_on='MSSA_ID', how='left')
merged = merged.drop(columns=['MSSA_ID'])

# ── Merge MSSA area ──────────────────────────────────────────────────────
merged = merged.merge(mssa_area, on='AreaID', how='left')

# ── Filter to lung cancer and drop unnecessary columns ───────────────────
merged = merged[merged['Cancer'] == 'Lung']
merged = merged.drop(['Counties', 'Cities', 'URL'], axis=1)

# ── Save the final merged dataset ────────────────────────────────────────
merged.to_csv(OUT_PATH, index=False)

# ── Summary ──────────────────────────────────────────────────────────────
print(f"Saved: {OUT_PATH}")
print(f"Rows: {merged.shape[0]:,}  |  Columns: {merged.shape[1]:,}")
print(f"Pollution/indicator columns (pop-weighted): {len(pollution_cols)}")
print(f"Smoking coverage: {merged['PerSmoking'].notna().sum()}/{len(merged)} rows")
print(f"Unique MSSAs: {merged['AreaID'].nunique()}")

Saved: /home/jovyan/CREATE Project/Data/healthmaps_with_ces4_pollution_plus_area.csv
Rows: 1,626  |  Columns: 63
Pollution/indicator columns (pop-weighted): 22
Smoking coverage: 1557/1626 rows
Unique MSSAs: 542


### 4a. Inspect the Merged Dataset

Display the first few rows to verify the merge and confirm column names
and data types.

In [5]:
merged.head()

,AreaID,Sex,Cancer,Years,PopTot,AAIR,LCI,UCI,White_PopTot,White_AAIR,...,CES_Asthma,CES_Low_Birth_Weight,CES_Cardiovascular_Disease,CES_Education,CES_Linguistic_Isolation,CES_Poverty,CES_Unemployment,CES_Housing_Burden,PerSmoking,MSSA_Area_SqMi
2168,1.1,Male,Lung,10yr,289444.0,45.7,37.7,54.6,173353.0,46.8,...,30.078710,4.391956,10.480837,6.922143,3.000305,11.802432,3.458344,13.111331,10.205207,1.417793
2169,1.2,Male,Lung,10yr,199372.0,39.6,30.5,49.8,124549.0,45.0,...,27.526394,3.838567,9.436112,5.212112,4.874291,9.436779,2.539407,6.438411,9.078531,1.843737
2170,10,Male,Lung,10yr,238669.0,76.8,67.2,87.0,157398.0,81.5,...,56.026987,4.551007,22.505578,16.717672,3.246941,49.882830,9.406186,15.276161,17.577372,2.914820
2171,100,Male,Lung,10yr,NaN,NaN,NaN,NaN,NaN,NaN,...,22.750000,NaN,9.170000,22.800000,2.500000,53.300000,NaN,9.100000,16.000000,956.060637
2172,102,Male,Lung,10yr,NaN,NaN,NaN,NaN,NaN,NaN,...,31.613504,4.994017,9.668119,8.514346,1.353135,27.075346,NaN,7.304038,13.662221,1452.205373
